In [17]:
import geopandas as gpd
import pandas as pd 
import numpy as np
import rasterio

from rasterio.features import rasterize
from rasterio.transform import from_origin


In [3]:
gdf = gpd.read_file('../data/input_datasets/Deforestation_Drivers_Data_archive_from_Camilo/PER_cacao_point.shp')

In [25]:

def rasterize_points(gdf, out_path, pixel_size=1.0, value=1):
    """
    Rasterize a set of points given as a GeoDataFrame.

    Parameters:
    gdf (GeoDataFrame): GeoDataFrame containing the points to rasterize.
    out_path (str): Path to save the output raster file.
    pixel_size (float): The pixel size of the output raster.
    value (int or float): The value to assign to the rasterized points.

    Returns:
    None
    """
    # Get the bounds of the GeoDataFrame
    bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
    
    # Define the raster size and transform
    width = int((bounds[2] - bounds[0]) / pixel_size)
    height = int((bounds[3] - bounds[1]) / pixel_size)
    transform = from_origin(bounds[0], bounds[3], pixel_size, pixel_size)
    
    # Create an empty raster
    raster = rasterio.open(
        out_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='int32',
        compress='lzw',  # or 'deflate', 'zstd'

        crs=gdf.crs.to_string(),
        transform=transform
    )
    
    # Generate (geometry, value) pairs
    shapes = ((geom, value) for geom in gdf.geometry)
    
    # Rasterize the points
    rasterized = rasterize(shapes=shapes, out_shape=(height, width), transform=transform, fill=0)
    
    # Write the raster to the file
    raster.write(rasterized, 1)
    raster.close()

In [22]:
gdf = gdf.to_crs(gdf.estimate_utm_crs())

In [28]:
rasterize_points(gdf, '../data/tmp/cocoa_peru_lzw.tif', pixel_size=100, value=1)